# Derive models of drone

Do all imports.

In [ ]:
import sympy as sym
import numpy as np

# Suppress the use of scientific notation when printing small numbers
np.set_printoptions(suppress=True)

## Dynamic model

Define physical parameters.

In [ ]:
params = {
    'm': 0.5,
    'Jx': 0.0023,
    'Jy': 0.0023,
    'Jz': 0.0040,
    'l': 0.175,
    'g': 9.81,
}

Derive the equations of motion:

In [ ]:
# components of position (meters)
p_x, p_y, p_z = sym.symbols('p_x, p_y, p_z')

# yaw, pitch, roll angles (radians)
psi, theta, phi = sym.symbols('psi, theta, phi')

# components of linear velocity (meters / second)
v_x, v_y, v_z = sym.symbols('v_x, v_y, v_z')
v_in_body = sym.Matrix([v_x, v_y, v_z])

# components of angular velocity (radians / second)
w_x, w_y, w_z = sym.symbols('w_x, w_y, w_z')
w_in_body = sym.Matrix([w_x, w_y, w_z])

# components of net rotor torque
tau_x, tau_y, tau_z = sym.symbols('tau_x, tau_y, tau_z')

# net rotor force
f_z = sym.symbols('f_z')

# parameters
m = sym.nsimplify(params['m'])
Jx = sym.nsimplify(params['Jx'])
Jy = sym.nsimplify(params['Jy'])
Jz = sym.nsimplify(params['Jz'])
l = sym.nsimplify(params['l'])
g = sym.nsimplify(params['g'])
J = sym.diag(Jx, Jy, Jz)

# rotation matrices
Rz = sym.Matrix([[sym.cos(psi), -sym.sin(psi), 0], [sym.sin(psi), sym.cos(psi), 0], [0, 0, 1]])
Ry = sym.Matrix([[sym.cos(theta), 0, sym.sin(theta)], [0, 1, 0], [-sym.sin(theta), 0, sym.cos(theta)]])
Rx = sym.Matrix([[1, 0, 0], [0, sym.cos(phi), -sym.sin(phi)], [0, sym.sin(phi), sym.cos(phi)]])
R_body_in_world = Rz @ Ry @ Rx

# angular velocity to angular rates
ex = sym.Matrix([[1], [0], [0]])
ey = sym.Matrix([[0], [1], [0]])
ez = sym.Matrix([[0], [0], [1]])
M = sym.simplify(sym.Matrix.hstack((Ry @ Rx).T @ ez, Rx.T @ ey, ex).inv(), full=True)

# applied forces
f_in_body = R_body_in_world.T @ sym.Matrix([[0], [0], [-m * g]]) + sym.Matrix([[0], [0], [f_z]])

# applied torques
tau_in_body = sym.Matrix([[tau_x], [tau_y], [tau_z]])

# equations of motion
f = sym.Matrix.vstack(
    R_body_in_world @ v_in_body,
    M @ w_in_body,
    (1 / m) * (f_in_body - w_in_body.cross(m * v_in_body)),
    J.inv() @ (tau_in_body - w_in_body.cross(J @ w_in_body)),
)

f = sym.simplify(f, full=True)

The equations of motion have this form:

$$\begin{bmatrix} \dot{p}_x \\ \dot{p}_y \\ \dot{p}_z \\ \dot{\psi} \\ \dot{\theta} \\ \dot{\phi} \\ \dot{v}_x \\ \dot{v}_y \\ \dot{v}_z \\ \dot{w}_x \\ \dot{w}_y \\ \dot{w}_z \end{bmatrix} = f\left(p_x, p_y, p_z, \psi, \theta, \phi, v_x, v_y, v_z, w_x, w_y, w_z, \tau_x, \tau_y, \tau_z, f_z \right)$$

Here is the function $f$:

In [ ]:
f

## Sensor model

Define the sensor model.

In [ ]:
# Position of drone in world frame
p_in_world = sym.Matrix([p_x, p_y, p_z])

# Position of markers in body frame
a_in_body = sym.Matrix([l, 0, 0])  # <-- marker on front rotor
b_in_body = sym.Matrix([-l, 0, 0]) # <-- marker on back rotor

# Position of markers in world frame
a_in_world = p_in_world + R_body_in_world @ a_in_body
b_in_world = p_in_world + R_body_in_world @ b_in_body

# Sensor model
g = sym.simplify(sym.Matrix.vstack(a_in_world, b_in_world))

The sensor model has this form:

$$o = g(p_x, p_y, p_z, \psi, \theta, \phi)$$

Here is the function $g$:

In [ ]:
g

## Model-based controller and observer design

### Linearization

Define the state and input of the nonlinear system:

$$
m = \begin{bmatrix} p_x \\ p_y \\ p_z \\ \psi \\ \theta \\ \phi \\ v_x \\ v_y \\ v_z \\ w_x \\ w_y \\ w_z \end{bmatrix}
\qquad\qquad
n = \begin{bmatrix} \tau_x \\ \tau_y \\ \tau_z \\ f_z \end{bmatrix}.
$$

In [ ]:
# FIXME (1)
#
# List all elements of the state (of the nonlinear
# system) as symbolic variables
m = []

# FIXME (2)
#
# List all elements of the input (of the nonlinear
# system) as symbolic variables
n = []

Choose an equilibrium point $m_e$ and $n_e$ that corresponds to hover at the origin with a zero yaw angle and that satisfies

$$ 0 = f(m_e, n_e).$$

In [ ]:
# FIXME (3)
#
# Convert `f` into a lambda function that can be
# evaluated for different choices of m_e and n_e
#
# It may be helpful to know that, if m and n are
# lists of symbolic variables, then
# 
#   m + n
#
# is a list that contains both the elements of m
# and the elements of n. (This is an easy way to
# define the arguments of the lambda function.)
f_num = None

# FIXME (4)
#
# Choose an equilibrium point.
#
# - List the equilibrium value of all elements of
#   the state (of the nonlinear system) as numbers
m_e = []
#
# - List the equilibrium value of all elements of
#   the input (of the nonlinear system) as numbers
n_e = []

# Check if m_e, n_e really is an equilibrium point.
#
# As above, if m_e and n_e are lists of numbers, then
#
#   m_e + n_e
#
# is a list that contains both the elements of m_e and
# the elements of n_e. Furthermore, if you want to pass
# all elements of m_e + n_e to a function as arguments,
# you can do this by "iterable unpacking" as follows:
#
#   *(m_e + n_e)
#
# See the documentation for more information:
#
#   https://docs.python.org/3/reference/expressions.html#expression-lists
#
# As an example, if
#
#   m_e = [1., 2.]
#   n_e = [3.]
#
# then
#
#   f_num(*(m_e + n_e))
#
# is equivalent to
#
#   f_num(1., 2., 3.)
#
# This check has been implemented for you - you only need
# to have defined f_num, m_e, and n_e previously.
assert(f_num(*(m_e + n_e)))

Linearize the nonlinear dynamic model

$$ \dot{m} = f(m, n) $$

to produce a linear dynamic model

$$ \dot{x} = A x + B u $$

where

$$
A = \frac{\partial f}{\partial m}\biggr\rvert_{(m_e, n_e)}
\qquad\qquad
B = \frac{\partial f}{\partial n}\biggr\rvert_{(m_e, n_e)}
$$

and

$$
x = m - m_e
\qquad\qquad
u = n - n_e.
$$

In [ ]:
# FIXME (5)
#
# Create lambda functions to evaluate the Jacobian of f with
# respect to m and n at different values of m_e and n_e.
#
# (Would "m + n" be helpful again here?)
#
A_num = None
B_num = None

# FIXME (6)
#
# Evaluate these lambda functions at the values of m_e and
# n_e that you chose.
#
# (Would "*(m_e + n_e)" be helpful again here?)
#
A = None
B = None

Linearize the nonlinear sensor model

$$ o = g(m, n) $$

to produce a linear sensor model

$$ y = Cx + Du $$

where

$$
C = \frac{\partial f}{\partial m}\biggr\rvert_{(m_e, n_e)}
\qquad\qquad
D = \frac{\partial f}{\partial n}\biggr\rvert_{(m_e, n_e)}
$$

and

$$
x = m - m_e
\qquad\qquad
u = n - n_e
\qquad\qquad
y = o - g(m_e, n_e).
$$

(You may find that $D = 0$ and so that part of your linear sensor model can be ignored.)

In [ ]:
# FIXME (7)
#
# Create lambda functions to evaluate the Jacobian of g with
# respect to m and n at different values of m_e and n_e.
C_num = None
D_num = None

# FIXME (8)
#
# Evaluate these lambda functions at the values of m_e and
# n_e that you chose.
C = None
D = None

# FIXME (9)
#
# Convert `g` into a lambda function that can be
# evaluated for different choices of m_e and n_e.
g_num = None

# FIXME (10)
#
# Find the sensor measurements
#
#   g(m_e, n_e)
#
# that you would expect to see at equilibrium. Call
# these sensor measurements o_e.
o_e = None

Make sure that `m_e`, `n_e`, and `o_e` are all 1D numpy arrays. (Until now, it was likely easier to work with `m_e` and `n_e` as lists instead of as arrays, for example.)

In [ ]:
m_e = np.array(m_e)
n_e = np.array(n_e)
o_e = np.array(o_e)

assert(m_e.ndim == 1)
assert(n_e.ndim == 1)
assert(o_e.ndim == 1)

### Controller design

Check that the system is controllable.

In [ ]:
# FIXME (optional for today, but required for the final report)
#
# Verify that the controllability matrix is full rank.

Define a function that solves the continuous-time, infinite-horizon LQR problem.

(You can find an implementation of this function at the [aerospace control systems reference page](https://aero.refpages.org/control/) at the end of the "Optimal controllers" section.)

In [ ]:
# FIXME (11)
def lqr(A, B, Q, R):
    pass

Choose weights $Q_c$ and $R_c$. Remember that $Q_c$ is a diagonal, square matrix of size $n_\text{x} \times n_\text{x}$ where $n_x$ is the number of states. Similarly, $R_c$ is a diagonal, square matrix of size $n_\text{u} \times n_\text{u}$ where $n_u$ is the number of inputs.

In [ ]:
# FIXME (12)
Q_c = None
R_c = None

Find the gain matrix $K$ that solves the optimal controller design problem corresponding to your choice of $Q_c$ and $R_c$.

In [ ]:
# FIXME (13)
K = None

### Observer design

Check that the system is observable.

In [ ]:
# FIXME (optional for today, but required for the final report)
#
# Verify that the observability matrix is full rank.

Choose weights $Q_o$ and $R_o$. Remember that $Q_o$ is a diagonal, square matrix of size $n_\text{y} \times n_\text{y}$ where $n_y$ is the number of outputs. Similarly, $R_o$ is a diagonal, square matrix of size $n_\text{x} \times n_\text{x}$ where $n_x$ is the number of states.

In [ ]:
# FIXME (14)
Q_o = None
R_o = None

Find the gain matrix $L$ that solves the optimal observer design problem corresponding to your choice of $Q_o$ and $R_o$.

(The end of the "Optimal observers" section of the [aerospace control systems reference page](https://aero.refpages.org/control/) reminds you how to use the same `lqr` function you defined nefore to solve the optimal observer design problem — the syntax is a little different than for solving the optimal controller design problem.)

In [ ]:
# FIXME (15)
L = None

### Make it easy to copy/paste your design into a `Controller` implementation

Define a function that rounds all elements of a numpy array to a chosen number of digits after the decimal (there is no need to keep around too many significant figures) and converts the result to a list so it can be printed compactly.

In [ ]:
def rnd(a, decimals=3):
    return np.round(a, decimals=decimals).tolist()

Print lines of text to copy/paste into a `Controller` implementation (if desired).

In [ ]:
print(f'        self.A = np.array({rnd(A)})')
print(f'        self.B = np.array({rnd(B)})')
print(f'        self.C = np.array({rnd(C)})')
print(f'        self.K = np.array({rnd(K)})')
print(f'        self.L = np.array({rnd(L)})')
print(f'        self.m_e = np.array({rnd(m_e)})')
print(f'        self.n_e = np.array({rnd(n_e)})')
print(f'        self.o_e = np.array({rnd(o_e)})')